In [1]:
from env import SimpleARGEnvironment
from utils import load_sequences

In [2]:
Ne = 10000
r_per_bp = 2e-8
mu_per_bp = 2e-8

dataset_path="validation/fasta/sim_l25kb_0.fa"

sequences = load_sequences(dataset_path)

In [3]:
env = SimpleARGEnvironment(
    num_sequences=len(sequences),
    population_size=Ne,
    recombination_rate=r_per_bp,
    mutation_rate=mu_per_bp,
    sequences=sequences,
    seed=7,
    bp_per_blocks=1
)

episodes = 1

In [8]:
import numpy as np
trajs = env.sample_log_rewards(2)
max_reward_seen = np.max(trajs)
init_Z = max_reward_seen

Sampling prior trajectory 1/2 for log Z init...
Sampling prior trajectory 2/2 for log Z init...


In [10]:
from tb_gfn import TBGFlowNetGenerator
from rollout_worker_arg import RolloutWorker
generator = TBGFlowNetGenerator(
        env,
        init_z_sample_count=2,
        device="cpu",
        verbose=True,
        policy_lr=0.001,
        log_z_lr=0.001,
        grad_clip=100,
        model_kwargs={},
    )
print(f"Generator device: {generator.device}")

rollout_worker = RolloutWorker(env)

verbose: True
Sampling prior trajectory 1/2 for log Z init...
Sampling prior trajectory 2/2 for log Z init...
Generator device: cpu


/Users/pratik/Documents/work/aim3/simpliied/arg/tb_gfn.py:96: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/opt/anaconda3/envs/phylogfn/lib/python3.10/site-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


In [11]:
ret, trajectories = rollout_worker.rollout(
            generator,
            episodes=1,
        )

In [12]:
import torch
log_paths_pf = ret['log_paths_pf']
log_paths_pb = ret['log_paths_pb']
log_rewards = torch.as_tensor(
    ret['log_rewards'],
    dtype=log_paths_pf.dtype,
    device=log_paths_pf.device,
)

log_pf = log_paths_pf.sum(-1)
log_pb = log_paths_pb.sum(-1)


log_z = generator.compute_log_Z(None).reshape(-1).to(log_paths_pf)

forward_value = log_z + log_pf
backward_value = log_rewards + log_pb

loss = generator.loss_fn(forward_value, backward_value)


In [18]:
import torch
# Compute the MSE loss between forward_value and backward_value from scratch
mse_loss = ((forward_value - backward_value) ** 2).mean()
print("MSE loss:", mse_loss.item(), forward_value - backward_value)

MSE loss: 1401199.375 tensor([-1183.7227], grad_fn=<SubBackward0>)


In [15]:
forward_value, log_z, log_pf, log_pb, log_rewards

(tensor([-9384.9746], grad_fn=<AddBackward0>),
 tensor([-7599.3677], grad_fn=<ViewBackward0>),
 tensor([-1785.6072], grad_fn=<SumBackward1>),
 tensor([0.]),
 tensor([-8201.2520]))

In [ ]:
a = [state[1] for state in trajs[0].transitions]

In [ ]:
for i in a:
    if i.is_done:
        state = i
        print(i.log_reward)
        break


In [ ]:
env.evolution_model.compute_arg_log_likelihood(state) + state.accumulated_log_prior + 33000

In [ ]:
from tb_gfn import TBGFlowNetGenerator
from rollout_worker_arg import RolloutWorker

generator = TBGFlowNetGenerator(env, 1)
model = generator.arg_model
rollout_worker = RolloutWorker(env)

In [ ]:
ret, trajectories = rollout_worker.rollout(generator, episodes=1)

In [ ]:
t = 0
s = trajectories[0].transitions[t][0]
s_next = trajectories[0].transitions[t][1]
a = trajectories[0].actions[t]

In [ ]:
import math 
log_pf = ret["log_paths_pf"][0, t].item()
num_parents = generator.count_backward_parents(s_next)
log_pb = -math.log(num_parents)

print("action:", a)
print("PF:", math.exp(log_pf))
print("PB:", math.exp(log_pb))
print("logPF:", log_pf)
print("logPB:", log_pb)
print("logPF - logPB:", log_pf - log_pb)

In [ ]:
log_pf = ret["log_paths_pf"][0].sum()
log_pb = ret["log_paths_pb"][0].sum()
log_r = ret["log_rewards"][0]
log_z = generator.compute_log_Z().detach()

residual = log_z + log_pf - (log_r + log_pb)
print(residual.item())

In [ ]:
log_r

In [ ]:
from env import Trajectory
states = [env.get_initial_state() for _ in range(episodes)]
trajectories = [Trajectory(x) for x in states]

unfinished = [idx for idx, state in enumerate(states) if not state.is_done]
active_states = [states[idx] for idx in unfinished]


input_dict = env.prepare_state_rollout_inputs(
    active_states,
    random_spec=None
    )

In [ ]:
## Encode states

states = input_dict["states"]
active_counts = [len(state.active_lineages) for state in states]

lineage = states[0].active_lineages[0]
feature = lineage.partials

In [ ]:
weights = model._material_segments_masking(lineage.material_segments, device=env.device, dtype=env.seq_arrays.dtype)

In [ ]:
masked_feature = feature * weights[:, None]

In [ ]:
normalize_feature = env.evolution_model.normalize_partials(masked_feature)

In [ ]:
normalize_feature